# v21 — ADX Bertingkat + Break of Structure (BOS) + Fibonacci S/R

**Latar belakang:** Investigasi langsung ke MT5 (data live, 21 Agustus 2026) membuktikan **ADX itu
lagging** -- pada beberapa kejadian breakout nyata (harga bergerak signifikan & konsisten 1 arah),
ADX masih rendah atau bahkan MENURUN, baru "mengejar" naik 45-60 menit KEMUDIAN. Contoh paling
ekstrem: jam 18:00-18:15 UTC (21 Agustus), ADX turun dari 16.3 ke 12.9 (di BAWAH filter dasar
v13 `adx_min=18`) sementara harga naik konsisten +0.2-0.27% berkali-kali -- robot v13 SAMA SEKALI
tidak bisa entry di momen ini krn ADX belum lolos filter, padahal harga sudah jelas bergerak.

**Tapi** ADX untuk trend yang SUDAH established (bukan baru mulai) tetap valid & tidak perlu
diubah -- v13 sudah tervalidasi kuat (v16: PF 3-4+) di kondisi ADX tinggi/trending jelas.

**Ide user**: robot butuh "baca pola" dulu sebelum menentukan level ADX yang dipakai --
1. **Trend FULL (ADX tinggi, established)** -> logika v13 skrg (ADX + momentum chain) SUDAH BENAR,
   tidak diubah.
2. **Awal breakout (ADX masih rendah TAPI harga sudah bergerak jelas)** -> perlu sinyal TAMBAHAN
   yang tidak bergantung ADX -- pakai **Break of Structure (BOS)**, indikator price-action murni
   (harga tembus swing high/low N candle), sudah ada di codebase (`bos_choch.py`), TIDAK ada bug
   lookahead (cuma `.shift(1)`, aman dipakai live).
3. **Ranging (ADX rendah, TIDAK ada breakout)** -> robot butuh baca Support/Resistance biar tidak
   "buta arah" -- pakai **Fibonacci retracement** (swing 50 candle, sudah ada di `fibonacci.py`)
   sbg area S/R dinamis: dekat support -> potensi mantul naik, dekat resistance -> potensi mantul
   turun.

**PENTING (instruksi eksplisit user)**: ini MURNI riset backtest di notebook. TIDAK ADA perubahan
ke `usecase.py` sampai hasil backtest v21 ini divalidasi & disetujui secara terpisah.

**Rencana kerja**:
1. Baca pola dulu: seberapa sering & seberapa signifikan kejadian "ADX telat, BOS duluan muncul"
2. Baseline: performa v13 murni (sbg pembanding, sudah ada dari riset sebelumnya)
3. Eksperimen A: tambah BOS sbg entry tambahan di kondisi ADX rendah (belum lolos filter v13
   biasa) -- apakah entry BOS-only ini profitable berdiri sendiri?
4. Eksperimen B: entry mean-reversion di kondisi ranging (ADX rendah, TANPA BOS) berbasis
   Fibonacci S/R -- beli dekat fib support, jual dekat fib resistance
5. TRAIN/TEST split & kriteria kejujuran yg sama spt v19/v20 (PF>1.5, sample TRAIN>=30, TEST>=15)

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STRATEGY_NAME = "m5_scalping"
VERSION = "v21"

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed" / STRATEGY_NAME
EXPORT_DIR = PROJECT_ROOT / "dataset" / "exports" / STRATEGY_NAME / VERSION
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
(PROCESSED_DIR / VERSION).mkdir(parents=True, exist_ok=True)

INITIAL_EQUITY = 100.0
RISK_PCT = 0.01
CONTRACT_SIZE = 100.0
MIN_LOT = 0.01
LOT_STEP = 0.01
REAL_SPREAD = 1.82  # spread broker MIFX riil, dari v18

MIN_SAMPLE_TRAIN = 30
MIN_SAMPLE_TEST = 15

pd.set_option("display.width", 160)
plt.rcParams["figure.figsize"] = (14, 5)

## 1. Load data 2025-2026 + kolom BOS/Fibonacci (sudah ada dari v01)

In [2]:
SCORED_CACHE_PATH = PROCESSED_DIR / VERSION / "df_2025_2026_v12_full_nofilter.parquet"
(PROCESSED_DIR / VERSION).mkdir(parents=True, exist_ok=True)

if SCORED_CACHE_PATH.exists():
    print(f"Load dari cache: {SCORED_CACHE_PATH}")
    df = pd.read_parquet(SCORED_CACHE_PATH)
else:
    print("Belum ada cache -- scoring v12 PENUH (semua 20 kategori, TANPA filter ADX/ATR pre-applied)...")
    print("PENTING: pakai score_de_redundant ASLI (bukan reimplementasi), sama persis dgn v13 live/backtest.")
    import time as _time
    from app.utils.signals.scoring_v12 import score_de_redundant

    df_m5 = pd.read_csv(PROCESSED_DIR / "v01" / "xauusd_m5_full_indicators.csv")
    df_m5["datetime"] = pd.to_datetime(df_m5["datetime"])
    df_m5 = df_m5.sort_values("datetime").reset_index(drop=True)
    df_m5 = df_m5[df_m5["datetime"] >= pd.Timestamp("2025-01-01", tz="UTC")].reset_index(drop=True)

    df_h1 = pd.read_csv(PROCESSED_DIR / "v01" / "xauusd_h1_full_indicators.csv")
    df_h1["datetime"] = pd.to_datetime(df_h1["datetime"])
    df_h1 = df_h1.sort_values("datetime").reset_index(drop=True)

    df_h1_shifted = df_h1.copy()
    df_h1_shifted["h1_available_at"] = df_h1_shifted["datetime"] + pd.Timedelta(hours=1)
    h1_cols = [c for c in df_h1_shifted.columns if c != "h1_available_at"]
    df_h1_shifted = df_h1_shifted[["h1_available_at", *h1_cols]].rename(columns={c: f"h1_{c}" for c in h1_cols})

    df_merged = pd.merge_asof(
        df_m5.sort_values("datetime"), df_h1_shifted.sort_values("h1_available_at"),
        left_on="datetime", right_on="h1_available_at", direction="backward",
    )
    print(f"Dataset 2025-2026 merged M5+H1: {len(df_merged)} baris")

    t0 = _time.time()
    scores = np.full(len(df_merged), np.nan)
    for pos in range(len(df_merged)):
        row = df_merged.iloc[pos]
        score, _breakdown = score_de_redundant(row)
        scores[pos] = score
        if pos % 20_000 == 0:
            print(f"  progress: {pos}/{len(df_merged)} ({_time.time()-t0:.0f}s)")

    df_merged["v12_score"] = scores
    print(f"Scoring selesai dalam {_time.time()-t0:.0f}s")

    keep_cols = [
        "datetime", "open", "high", "low", "close", "adx", "atr", "v12_score",
        "bull_chain", "bear_chain",
        "bos_bull", "bos_bear", "fib_swing_high", "fib_swing_low",
        "fib_236", "fib_382", "fib_500", "fib_618", "fib_786",
        "ob_bull", "ob_bear", "h1_ob_bull", "h1_ob_bear",
        "h1_ema_50", "h1_ema_200",
    ]
    keep_cols = [c for c in keep_cols if c in df_merged.columns]
    df = df_merged[keep_cols].copy()
    df.to_parquet(SCORED_CACHE_PATH, index=False)
    print(f"Tersimpan ke cache: {SCORED_CACHE_PATH}")

needed_cols = ["datetime", "open", "high", "low", "close", "adx", "atr", "v12_score",
               "bull_chain", "bear_chain", "bos_bull", "bos_bear",
               "fib_swing_high", "fib_swing_low", "fib_236", "fib_382", "fib_500", "fib_618", "fib_786",
               "ob_bull", "ob_bear", "h1_ob_bull", "h1_ob_bear", "h1_ema_50", "h1_ema_200"]
missing = [c for c in needed_cols if c not in df.columns]
print("Kolom hilang:", missing)
print(f"\nTotal candle: {len(df)}, {df['datetime'].min()} -> {df['datetime'].max()}")

Belum ada cache -- scoring v12 PENUH (semua 20 kategori, TANPA filter ADX/ATR pre-applied)...
PENTING: pakai score_de_redundant ASLI (bukan reimplementasi), sama persis dgn v13 live/backtest.


Dataset 2025-2026 merged M5+H1: 107336 baris
  progress: 0/107336 (0s)


  progress: 20000/107336 (3s)


  progress: 40000/107336 (7s)


  progress: 60000/107336 (10s)


  progress: 80000/107336 (13s)


  progress: 100000/107336 (16s)


Scoring selesai dalam 17s


Tersimpan ke cache: D:\Projects\robot-scalping\dataset\processed\m5_scalping\v21\df_2025_2026_v12_full_nofilter.parquet
Kolom hilang: []

Total candle: 107336, 2025-01-01 23:00:00+00:00 -> 2026-08-06 12:35:00+00:00


## 2. Baca pola: seberapa sering BOS muncul SEBELUM ADX "mengejar" naik?

Definisi kejadian target: candle dimana `bos_bull`/`bos_bear` = 1 (breakout struktur baru saja
terjadi), TAPI `adx` saat itu masih < 25 (belum dianggap "trending" oleh definisi v13/v16). Cek
apakah ADX benar2 naik signifikan dalam N candle berikutnya (mengejar), dan seberapa besar
pergerakan harga yg terjadi SEBELUM ADX itu mengejar (ini "peluang hilang" v13 saat ini).

In [3]:
LOOKFORWARD = 12  # 1 jam M5, konsisten dgn max_hold v13

df["adx_future_max"] = df["adx"].shift(-1).rolling(LOOKFORWARD, min_periods=1).max().shift(-(LOOKFORWARD - 1))
df["close_future"] = df["close"].shift(-LOOKFORWARD)
df["fwd_move_pct"] = (df["close_future"] - df["close"]) / df["close"] * 100

bos_early = df[((df["bos_bull"] == 1) | (df["bos_bear"] == 1)) & (df["adx"] < 25)].copy()
print(f"Total candle dgn BOS TAPI ADX<25 (potensi 'breakout awal yg ADX belum sadar'): {len(bos_early)}")
print(f"Dari total {len(df)} candle ({len(bos_early)/len(df)*100:.2f}%)")
print()

bos_early["adx_did_catch_up"] = bos_early["adx_future_max"] >= 25
print(f"Dari kejadian ini, ADX benar2 'mengejar' ke >=25 dalam {LOOKFORWARD} candle berikutnya: "
      f"{bos_early['adx_did_catch_up'].mean()*100:.1f}%")
print()
print("=== Pergerakan harga (fwd_move_pct) di kejadian BOS+ADX<25, dipecah BOS bull vs bear ===")
bull_events = bos_early[bos_early["bos_bull"] == 1]
bear_events = bos_early[bos_early["bos_bear"] == 1]
print(f"BOS BULL (n={len(bull_events)}): fwd_move_pct mean={bull_events['fwd_move_pct'].mean():.3f}%, "
      f"median={bull_events['fwd_move_pct'].median():.3f}%, %positif={(bull_events['fwd_move_pct']>0).mean()*100:.1f}%")
print(f"BOS BEAR (n={len(bear_events)}): fwd_move_pct mean={bear_events['fwd_move_pct'].mean():.3f}%, "
      f"median={bear_events['fwd_move_pct'].median():.3f}%, %negatif={(bear_events['fwd_move_pct']<0).mean()*100:.1f}%")

Total candle dgn BOS TAPI ADX<25 (potensi 'breakout awal yg ADX belum sadar'): 6733
Dari total 107336 candle (6.27%)

Dari kejadian ini, ADX benar2 'mengejar' ke >=25 dalam 12 candle berikutnya: 52.1%

=== Pergerakan harga (fwd_move_pct) di kejadian BOS+ADX<25, dipecah BOS bull vs bear ===
BOS BULL (n=3933): fwd_move_pct mean=0.004%, median=0.001%, %positif=50.0%
BOS BEAR (n=2800): fwd_move_pct mean=0.007%, median=0.018%, %negatif=46.2%


## 3. Backtest engine: v13 ASLI (v12_score + Order Block + H1 alignment) + 2 mode tambahan

**Perbaikan penting dari draft awal**: mode TREND sekarang memanggil skor v12 sungguhan
(`v12_score >= min_signal_score`, threshold 9.0 sama persis dgn v13 live) DAN Order Block filter
(`has_opposing_order_block`-equivalent) DAN H1 alignment -- BUKAN reimplementasi sederhana
`momentum_chain >= 5` yang terbukti salah (baseline jadi rugi, padahal v13 asli untung jelas).
Ini supaya Baseline A di Section 4 benar-benar SETARA v13 asli yang sudah divalidasi berkali-kali
(PF 3.57 di TEST) -- kalau baseline-nya salah, semua perbandingan BOS/Fib jadi tidak berarti.

Kondisi ENTRY per mode:
- **Mode TREND** (ADX>=adx_trend_min): v12_score>=9.0 (atau <=-9.0 utk SELL), TIDAK ada Order
  Block lawan arah, H1 EMA50 vs EMA200 searah (kalau data H1 tersedia)
- **Mode BOS-AWAL** (adx_ranging_max <= ADX < adx_trend_min, DAN bos_bull/bos_bear baru muncul):
  entry searah BOS, SL/TP ATR-relatif -- 'menangkap' breakout sblm ADX sempat mengejar
- **Mode FIB-RANGING** (ADX < adx_ranging_max): entry mean-reversion dekat level fib
  support/resistance -- BUY dekat fib_618/786, SELL dekat fib_236

In [4]:
def check_h1_alignment_v21(h1_ema_50, h1_ema_200, direction: str) -> bool:
    if h1_ema_50 is None or h1_ema_200 is None or not np.isfinite(h1_ema_50) or not np.isfinite(h1_ema_200):
        return True  # data H1 tidak tersedia -- tidak memblokir (spt _check_htf_alignment asli)
    h1_trend = "UP" if h1_ema_50 > h1_ema_200 else ("DOWN" if h1_ema_50 < h1_ema_200 else "FLAT")
    if direction == "BUY" and h1_trend == "DOWN":
        return False
    if direction == "SELL" and h1_trend == "UP":
        return False
    return True


def run_backtest_adaptive(
    df_signals: pd.DataFrame,
    adx_trend_min: float,       # >= ini: mode TREND (v13-style, v12_score asli)
    adx_ranging_max: float,     # < ini: mode FIB-RANGING; di antara: mode BOS-AWAL
    min_signal_score: float,    # threshold v12_score, v13 asli pakai 9.0
    sl_mult_trend: float,
    tp_mult_trend: float,
    sl_mult_bos: float,
    tp_mult_bos: float,
    fib_proximity_pct: float,   # toleransi jarak ke level fib (% dari range swing)
    sl_mult_fib: float,
    max_hold: int,
    enable_trend: bool = True,
    enable_bos: bool = True,
    enable_fib: bool = True,
    require_ob_filter: bool = True,
    require_h1_alignment: bool = True,
    spread_points: float = REAL_SPREAD,
) -> pd.DataFrame:
    close_arr = df_signals["close"].to_numpy()
    high_arr = df_signals["high"].to_numpy()
    low_arr = df_signals["low"].to_numpy()
    adx_arr = df_signals["adx"].to_numpy()
    atr_arr = df_signals["atr"].to_numpy()
    score_arr = df_signals["v12_score"].to_numpy()
    bos_bull_arr = df_signals["bos_bull"].to_numpy()
    bos_bear_arr = df_signals["bos_bear"].to_numpy()
    ob_bull_arr = df_signals["ob_bull"].to_numpy() if "ob_bull" in df_signals.columns else np.zeros(len(df_signals))
    ob_bear_arr = df_signals["ob_bear"].to_numpy() if "ob_bear" in df_signals.columns else np.zeros(len(df_signals))
    h1_ob_bull_arr = df_signals["h1_ob_bull"].to_numpy() if "h1_ob_bull" in df_signals.columns else np.zeros(len(df_signals))
    h1_ob_bear_arr = df_signals["h1_ob_bear"].to_numpy() if "h1_ob_bear" in df_signals.columns else np.zeros(len(df_signals))
    h1_ema_50_arr = df_signals["h1_ema_50"].to_numpy() if "h1_ema_50" in df_signals.columns else np.full(len(df_signals), np.nan)
    h1_ema_200_arr = df_signals["h1_ema_200"].to_numpy() if "h1_ema_200" in df_signals.columns else np.full(len(df_signals), np.nan)
    fib_236 = df_signals["fib_236"].to_numpy()
    fib_618 = df_signals["fib_618"].to_numpy()
    fib_786 = df_signals["fib_786"].to_numpy()
    swing_high = df_signals["fib_swing_high"].to_numpy()
    swing_low = df_signals["fib_swing_low"].to_numpy()
    datetime_arr = df_signals["datetime"].to_numpy()
    n = len(df_signals)

    trades = []
    equity = INITIAL_EQUITY
    i = 0
    while i < n:
        adx, atr, close, score = adx_arr[i], atr_arr[i], close_arr[i], score_arr[i]
        if not np.isfinite(atr) or atr <= 0 or not np.isfinite(adx) or not np.isfinite(score):
            i += 1
            continue

        direction = None
        mode = None
        sl_mult = tp_mult = None

        if adx >= adx_trend_min and enable_trend:
            if score >= min_signal_score:
                direction = "BUY"
            elif score <= -min_signal_score:
                direction = "SELL"
            if direction is not None:
                if require_ob_filter:
                    opposing_ob = (
                        (direction == "BUY" and (ob_bear_arr[i] > 0 or h1_ob_bear_arr[i] > 0)) or
                        (direction == "SELL" and (ob_bull_arr[i] > 0 or h1_ob_bull_arr[i] > 0))
                    )
                    if opposing_ob:
                        direction = None
                if direction is not None and require_h1_alignment:
                    if not check_h1_alignment_v21(h1_ema_50_arr[i], h1_ema_200_arr[i], direction):
                        direction = None
            sl_mult, tp_mult = sl_mult_trend, tp_mult_trend

        elif adx_ranging_max <= adx < adx_trend_min and enable_bos:
            if bos_bull_arr[i] == 1:
                direction, mode = "BUY", "BOS"
            elif bos_bear_arr[i] == 1:
                direction, mode = "SELL", "BOS"
            sl_mult, tp_mult = sl_mult_bos, tp_mult_bos

        elif adx < adx_ranging_max and enable_fib:
            rng = swing_high[i] - swing_low[i]
            if np.isfinite(rng) and rng > 0:
                tol = rng * fib_proximity_pct
                near_support = abs(close - fib_786[i]) <= tol or abs(close - fib_618[i]) <= tol
                near_resistance = abs(close - fib_236[i]) <= tol
                if near_support:
                    direction, mode = "BUY", "FIB"
                    sl_mult, tp_mult = sl_mult_fib, None
                elif near_resistance:
                    direction, mode = "SELL", "FIB"
                    sl_mult, tp_mult = sl_mult_fib, None

        if direction is None:
            i += 1
            continue
        if mode is None:
            mode = "TREND"

        sl_points = sl_mult * atr
        entry_price = close + (spread_points if direction == "BUY" else -spread_points)
        if mode == "FIB":
            fib_mid = (swing_high[i] + swing_low[i]) / 2
            tp_price = fib_mid
            if (direction == "BUY" and tp_price <= entry_price) or (direction == "SELL" and tp_price >= entry_price):
                i += 1
                continue
        else:
            tp_points = tp_mult * atr
            tp_price = entry_price + tp_points if direction == "BUY" else entry_price - tp_points
        sl_price = entry_price - sl_points if direction == "BUY" else entry_price + sl_points

        entry_time = datetime_arr[i]
        exit_price = None
        exit_idx = min(i + max_hold, n - 1)
        window_end = min(i + 1 + max_hold, n)
        for candle_idx in range(i + 1, window_end):
            c_high, c_low = high_arr[candle_idx], low_arr[candle_idx]
            hit_tp = c_high >= tp_price if direction == "BUY" else c_low <= tp_price
            hit_sl = c_low <= sl_price if direction == "BUY" else c_high >= sl_price
            if hit_sl:
                exit_price, exit_time = sl_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
            if hit_tp:
                exit_price, exit_time = tp_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
        if exit_price is None:
            exit_price, exit_time = close_arr[exit_idx], datetime_arr[exit_idx]

        next_i = exit_idx + 1
        price_move = (exit_price - entry_price) if direction == "BUY" else (entry_price - exit_price)
        risk_amount = equity * RISK_PCT
        lot = max(round(math.floor((risk_amount / (sl_points * CONTRACT_SIZE)) / LOT_STEP) * LOT_STEP, 2), MIN_LOT) if sl_points > 0 else MIN_LOT
        pnl = price_move * lot * CONTRACT_SIZE
        equity += pnl
        trades.append({
            "entry_time": entry_time, "mode": mode, "direction": direction, "pnl": pnl,
            "result": "WIN" if pnl > 0 else "LOSS", "equity_after": equity,
        })
        i = next_i

    return pd.DataFrame(trades)


def evaluate(trades: pd.DataFrame, initial_equity: float) -> dict:
    if trades.empty:
        return {"total_trades": 0, "win_rate_pct": 0, "profit_factor": 0, "net_pnl": 0, "max_drawdown_pct": 0}
    wins = trades[trades["pnl"] > 0]
    losses = trades[trades["pnl"] <= 0]
    gross_profit = wins["pnl"].sum()
    gross_loss = losses["pnl"].sum()
    equity_series = pd.Series([initial_equity] + trades["equity_after"].tolist())
    running_max = equity_series.cummax()
    drawdown = (equity_series - running_max) / running_max * 100
    return {
        "total_trades": len(trades),
        "win_rate_pct": round(len(wins) / len(trades) * 100, 2),
        "profit_factor": round(gross_profit / abs(gross_loss), 2) if gross_loss != 0 else float("inf"),
        "net_pnl": round(gross_profit + gross_loss, 2),
        "max_drawdown_pct": round(drawdown.min(), 2),
    }


def evaluate_by_mode(trades: pd.DataFrame) -> pd.DataFrame:
    if trades.empty:
        return pd.DataFrame()
    rows = []
    for mode, g in trades.groupby("mode"):
        wins = (g["result"] == "WIN").sum()
        gp = g.loc[g["pnl"] > 0, "pnl"].sum()
        gl = abs(g.loc[g["pnl"] <= 0, "pnl"].sum())
        rows.append({
            "mode": mode, "n": len(g), "win_rate_pct": round(wins / len(g) * 100, 1),
            "pf": round(gp / gl, 2) if gl > 0 else float("inf"), "net_pnl": round(g["pnl"].sum(), 2),
        })
    return pd.DataFrame(rows)

print("Backtest engine adaptive (v13 asli + BOS + FIB) siap.")

Backtest engine adaptive (v13 asli + BOS + FIB) siap.


## 4. TRAIN/TEST split & baseline

Baseline A = v13 murni (cuma mode TREND, BOS & FIB dimatikan) -- pembanding utama.
Baseline B = v13 + BOS only (FIB dimatikan) -- isolasi kontribusi BOS.
Baseline C = v13 + FIB only (BOS dimatikan) -- isolasi kontribusi FIB.
Full = v13 + BOS + FIB (semua mode aktif).

In [5]:
TRAIN_END = pd.Timestamp("2026-03-01", tz="UTC")
df_train = df[df["datetime"] < TRAIN_END].reset_index(drop=True)
df_test = df[df["datetime"] >= TRAIN_END].reset_index(drop=True)
print(f"TRAIN: {len(df_train)} candle | TEST: {len(df_test)} candle")

base_params = dict(
    adx_trend_min=25.0, adx_ranging_max=18.0, min_signal_score=9.0,
    sl_mult_trend=2.0, tp_mult_trend=4.0,
    sl_mult_bos=2.0, tp_mult_bos=3.0,
    fib_proximity_pct=0.05, sl_mult_fib=1.5,
    max_hold=12,
)

print("\n=== Baseline A: TREND only (setara v13) ===")
trades_a_train = run_backtest_adaptive(df_train, enable_trend=True, enable_bos=False, enable_fib=False, **base_params)
trades_a_test = run_backtest_adaptive(df_test, enable_trend=True, enable_bos=False, enable_fib=False, **base_params)
print("TRAIN:", evaluate(trades_a_train, INITIAL_EQUITY))
print("TEST:", evaluate(trades_a_test, INITIAL_EQUITY))

print("\n=== Baseline B: TREND + BOS ===")
trades_b_train = run_backtest_adaptive(df_train, enable_trend=True, enable_bos=True, enable_fib=False, **base_params)
trades_b_test = run_backtest_adaptive(df_test, enable_trend=True, enable_bos=True, enable_fib=False, **base_params)
print("TRAIN:", evaluate(trades_b_train, INITIAL_EQUITY))
print(evaluate_by_mode(trades_b_train).to_string(index=False))
print("TEST:", evaluate(trades_b_test, INITIAL_EQUITY))
print(evaluate_by_mode(trades_b_test).to_string(index=False))

print("\n=== Baseline C: TREND + FIB ===")
trades_c_train = run_backtest_adaptive(df_train, enable_trend=True, enable_bos=False, enable_fib=True, **base_params)
trades_c_test = run_backtest_adaptive(df_test, enable_trend=True, enable_bos=False, enable_fib=True, **base_params)
print("TRAIN:", evaluate(trades_c_train, INITIAL_EQUITY))
print(evaluate_by_mode(trades_c_train).to_string(index=False))
print("TEST:", evaluate(trades_c_test, INITIAL_EQUITY))
print(evaluate_by_mode(trades_c_test).to_string(index=False))

print("\n=== Full: TREND + BOS + FIB ===")
trades_full_train = run_backtest_adaptive(df_train, enable_trend=True, enable_bos=True, enable_fib=True, **base_params)
trades_full_test = run_backtest_adaptive(df_test, enable_trend=True, enable_bos=True, enable_fib=True, **base_params)
print("TRAIN:", evaluate(trades_full_train, INITIAL_EQUITY))
print(evaluate_by_mode(trades_full_train).to_string(index=False))
print("TEST:", evaluate(trades_full_test, INITIAL_EQUITY))
print(evaluate_by_mode(trades_full_test).to_string(index=False))

TRAIN: 77226 candle | TEST: 30110 candle

=== Baseline A: TREND only (setara v13) ===


TRAIN: {'total_trades': 318, 'win_rate_pct': 49.37, 'profit_factor': np.float64(1.61), 'net_pnl': np.float64(503.27), 'max_drawdown_pct': np.float64(-44.08)}
TEST: {'total_trades': 121, 'win_rate_pct': 61.98, 'profit_factor': np.float64(2.63), 'net_pnl': np.float64(694.31), 'max_drawdown_pct': np.float64(-31.93)}

=== Baseline B: TREND + BOS ===


TRAIN: {'total_trades': 1921, 'win_rate_pct': 33.32, 'profit_factor': np.float64(0.68), 'net_pnl': np.float64(-2448.25), 'max_drawdown_pct': np.float64(-2452.75)}
 mode    n  win_rate_pct   pf  net_pnl
  BOS 1665          30.9 0.58 -2937.54
TREND  256          48.8 1.73   489.29
TEST: {'total_trades': 719, 'win_rate_pct': 39.78, 'profit_factor': np.float64(0.85), 'net_pnl': np.float64(-599.8), 'max_drawdown_pct': np.float64(-286.96)}
 mode   n  win_rate_pct   pf  net_pnl
  BOS 618          36.1 0.69 -1147.34
TREND 101          62.4 2.56   547.54

=== Baseline C: TREND + FIB ===


TRAIN: {'total_trades': 2812, 'win_rate_pct': 36.49, 'profit_factor': np.float64(0.5), 'net_pnl': np.float64(-4211.26), 'max_drawdown_pct': np.float64(-4211.26)}
 mode    n  win_rate_pct   pf  net_pnl
  FIB 2494          34.8 0.38 -4717.01
TREND  318          49.4 1.61   505.75
TEST: {'total_trades': 1152, 'win_rate_pct': 48.78, 'profit_factor': np.float64(0.72), 'net_pnl': np.float64(-1247.78), 'max_drawdown_pct': np.float64(-677.16)}
 mode    n  win_rate_pct   pf  net_pnl
  FIB 1031          47.2 0.52 -1942.09
TREND  121          62.0 2.63   694.31

=== Full: TREND + BOS + FIB ===


TRAIN: {'total_trades': 4279, 'win_rate_pct': 33.98, 'profit_factor': np.float64(0.52), 'net_pnl': np.float64(-7085.48), 'max_drawdown_pct': np.float64(-7085.48)}
 mode    n  win_rate_pct   pf  net_pnl
  BOS 1602          30.4 0.56 -3007.03
  FIB 2422          34.8 0.38 -4554.48
TREND  255          48.6 1.71   476.02
TEST: {'total_trades': 1677, 'win_rate_pct': 43.95, 'profit_factor': np.float64(0.67), 'net_pnl': np.float64(-2555.93), 'max_drawdown_pct': np.float64(-1870.69)}
 mode    n  win_rate_pct   pf  net_pnl
  BOS  571          34.9 0.66 -1185.68
  FIB 1003          47.2 0.51 -1938.90
TREND  103          63.1 2.69   568.65


## 5. Grid search -- cari kombinasi terbaik (kalau baseline default blm optimal)

In [6]:
import itertools
import time as _time

GRID = {
    "adx_trend_min": [22.0, 25.0, 28.0],
    "adx_ranging_max": [15.0, 18.0],
    "sl_mult_bos": [1.5, 2.0],
    "tp_mult_bos": [2.0, 3.0, 4.0],
    "fib_proximity_pct": [0.03, 0.05, 0.08],
    "sl_mult_fib": [1.0, 1.5, 2.0],
}
FIXED = dict(sl_mult_trend=2.0, tp_mult_trend=4.0, min_signal_score=9.0, max_hold=12, enable_trend=True, enable_bos=True, enable_fib=True)

combos = list(itertools.product(*GRID.values()))
print(f"Total kombinasi grid: {len(combos)}")

t0 = _time.time()
grid_results = []
for idx, combo in enumerate(combos):
    params = dict(zip(GRID.keys(), combo))
    trades = run_backtest_adaptive(df_train, **params, **FIXED)
    metrics = evaluate(trades, INITIAL_EQUITY)
    metrics.update(params)
    grid_results.append(metrics)
    if (idx + 1) % 50 == 0:
        print(f"  [{idx+1}/{len(combos)}] {_time.time()-t0:.0f}s")

grid_df = pd.DataFrame(grid_results)
print(f"\nGrid search selesai dalam {_time.time()-t0:.0f}s")

grid_valid = grid_df[grid_df["total_trades"] >= MIN_SAMPLE_TRAIN].sort_values("profit_factor", ascending=False)
print(f"\n=== Top 10 kandidat (sample TRAIN >= {MIN_SAMPLE_TRAIN}) ===")
print(grid_valid.head(10).to_string(index=False))

Total kombinasi grid: 324


  [50/324] 9s


  [100/324] 19s


  [150/324] 28s


  [200/324] 37s


  [250/324] 46s


  [300/324] 56s



Grid search selesai dalam 60s

=== Top 10 kandidat (sample TRAIN >= 30) ===
 total_trades  win_rate_pct  profit_factor  net_pnl  max_drawdown_pct  adx_trend_min  adx_ranging_max  sl_mult_bos  tp_mult_bos  fib_proximity_pct  sl_mult_fib
         2801         35.17           0.58 -4320.82          -4320.82           22.0             15.0          2.0          4.0               0.03          2.0
         2810         35.69           0.58 -4342.99          -4342.99           22.0             15.0          2.0          3.0               0.03          2.0
         2984         40.18           0.58 -4463.23          -4467.42           22.0             18.0          2.0          4.0               0.03          2.0
         3028         36.00           0.57 -4812.06          -4812.06           22.0             15.0          2.0          4.0               0.05          2.0
         2946         32.89           0.57 -4576.93          -4576.93           22.0             15.0          2.0         

## 6. Validasi TEST out-of-sample (kandidat terbaik grid search)

In [7]:
candidates_passing = grid_valid[grid_valid["profit_factor"] > grid_df.loc[grid_df.index.isin(
    grid_valid.index), "profit_factor"].median()]  # ambil kandidat di atas median dulu utk overview

best_row = grid_valid.iloc[0]
best_params = {k: best_row[k] for k in GRID.keys()}
print(f"Kandidat terbaik TRAIN: {best_params}")
print("TRAIN:", {k: best_row[k] for k in ["total_trades","win_rate_pct","profit_factor","net_pnl","max_drawdown_pct"]})

trades_test_best = run_backtest_adaptive(df_test, **best_params, **FIXED)
test_metrics = evaluate(trades_test_best, INITIAL_EQUITY)
print("\nTEST (out-of-sample):", test_metrics)
print(evaluate_by_mode(trades_test_best).to_string(index=False))

baseline_a_test_metrics = evaluate(trades_a_test, INITIAL_EQUITY)
print(f"\n=== Perbandingan TEST: Baseline A (v13 murni) vs kandidat terbaik ===")
print(f"Baseline A : {baseline_a_test_metrics}")
print(f"Kandidat   : {test_metrics}")

if test_metrics["total_trades"] < MIN_SAMPLE_TEST:
    print(f"\n>>> PERINGATAN: sample TEST ({test_metrics['total_trades']}) < minimum ({MIN_SAMPLE_TEST}).")
elif test_metrics["profit_factor"] > baseline_a_test_metrics["profit_factor"] and test_metrics["net_pnl"] > baseline_a_test_metrics["net_pnl"]:
    print("\n>>> Kandidat MENGUNGGULI baseline v13 murni di TEST (PF & net_pnl lebih baik) -- layak dipertimbangkan.")
else:
    print("\n>>> Kandidat TIDAK secara jelas mengungguli baseline v13 murni -- perlu evaluasi trade-off lebih detail (lihat evaluate_by_mode).")

Kandidat terbaik TRAIN: {'adx_trend_min': np.float64(22.0), 'adx_ranging_max': np.float64(15.0), 'sl_mult_bos': np.float64(2.0), 'tp_mult_bos': np.float64(4.0), 'fib_proximity_pct': np.float64(0.03), 'sl_mult_fib': np.float64(2.0)}
TRAIN: {'total_trades': np.float64(2801.0), 'win_rate_pct': np.float64(35.17), 'profit_factor': np.float64(0.58), 'net_pnl': np.float64(-4320.82), 'max_drawdown_pct': np.float64(-4320.82)}

TEST (out-of-sample): {'total_trades': 1062, 'win_rate_pct': 43.03, 'profit_factor': np.float64(0.74), 'net_pnl': np.float64(-1416.88), 'max_drawdown_pct': np.float64(-1000.25)}
 mode   n  win_rate_pct   pf  net_pnl
  BOS 561          33.3 0.69  -995.81
  FIB 377          52.3 0.43  -976.27
TREND 124          58.9 2.19   555.21

=== Perbandingan TEST: Baseline A (v13 murni) vs kandidat terbaik ===
Baseline A : {'total_trades': 121, 'win_rate_pct': 61.98, 'profit_factor': np.float64(2.63), 'net_pnl': np.float64(694.31), 'max_drawdown_pct': np.float64(-31.93)}
Kandidat   : 

## 7. Kesimpulan

*(diisi setelah lihat hasil eksekusi lengkap Section 2-6 -- placeholder)*